# Analiza sieciowa scenariusza filmowego

## Cel

Ten notatnik przeprowadza **analizę sieciową scenariusza filmowego** pobranego z serwisu [IMSDb](https://imsdb.com/). Efektem końcowym jest graf powiązań między postaciami — struktura danych pokazująca, kto z kim występuje w scenach i jak często.

Poszczególne etapy obejmują:
1. Pobranie tekstu scenariusza ze strony internetowej.
2. Wyodrębnienie scen i postaci.
3. Analizę częstotliwości występowania postaci.
4. Budowę tabeli współwystąpień (krawędzi grafu).
5. Wizualizację sieci i eksport danych do formatów czytelnych dla narzędzi takich jak Gephi.

## Czym jest vibe coding?

Każda komórka z kodem zaczyna się od **promptu** — opisu w języku naturalnym, który precyzyjnie definiuje *co* ma zostać zrobione i *jaki rezultat* jest oczekiwany. Prompt nie wskazuje konkretnych narzędzi ani bibliotek. To podejście nazywane jest **vibe codingiem**: zamiast pisać kod od zera, opisujesz cel, a sztuczna inteligencja generuje działające rozwiązanie.

Możesz skopiować dowolny prompt z tego notatnika, wkleić go do modelu AI (np. Claude, ChatGPT) i otrzymać alternatywną implementację — prawdopodobnie równie poprawną, choć napisaną inaczej.

---
## Etap 1: Pobranie danych

Scenariusze na IMSDb są zwykłymi stronami HTML. Tekst scenariusza znajduje się wewnątrz preformatowanego bloku (`<pre>`), otoczonego elementami nawigacyjnymi witryny, które nas nie interesują.

W tym kroku pobieramy zawartość strony i wyciągamy z niej wyłącznie czysty tekst scenariusza. Adres URL jest zapisany w zmiennej — wystarczy go podmienić, żeby przeanalizować inny film.

In [ ]:
"""
PROMPT DLA AI:
Napisz skrypt, który:
1. Przyjmuje adres URL strony ze scenariuszem filmowym z serwisu IMSDb.
2. Pobiera zawartość tej strony internetowej.
3. Ignoruje wszystkie elementy nawigacyjne witryny (menu, stopkę itp.).
4. Wyodrębnia wyłącznie czysty tekst scenariusza, który na tej stronie
   znajduje się wewnątrz preformatowanego bloku tekstu.
5. Wyświetla pierwszych 3000 znaków wyodrębnionego tekstu, aby można było
   zweryfikować poprawność pobrania.
Adres URL powinien być przechowywany w osobnej zmiennej, żeby łatwo go zmienić.
"""

import requests
from bs4 import BeautifulSoup

# === ZMIEŃ TEN ADRES URL NA DOWOLNY SCENARIUSZ Z IMSDb ===
URL_SCENARIUSZA = "https://imsdb.com/scripts/Pulp-Fiction.html"

# Pobranie zawartości strony
odpowiedz = requests.get(URL_SCENARIUSZA)
odpowiedz.raise_for_status()  # przerwij, jeśli strona nie odpowiada

# Wyodrębnienie tekstu z preformatowanego bloku <pre>
zupa = BeautifulSoup(odpowiedz.text, "html.parser")
blok_pre = zupa.find("pre")

if blok_pre is None:
    raise ValueError("Nie znaleziono bloku <pre> na stronie. Sprawdź adres URL.")

tekst_scenariusza = blok_pre.get_text()

# Podgląd pierwszych 3000 znaków
print(f"Pobrano {len(tekst_scenariusza)} znaków tekstu scenariusza.")
print("=" * 60)
print(tekst_scenariusza[:3000])

---
## Etap 2: Parsowanie struktury — sceny i postacie

Scenariusz filmowy ma powtarzalną konwencję formatowania:

- **Nagłówki scen** zaczynają się od słów kluczowych `INT.` (wnętrze) lub `EXT.` (plener).
- **Nazwy postaci** pojawiają się jako tekst pisany WIELKIMI LITERAMI, wyraźnie wcięty od lewego marginesu (zazwyczaj co najmniej 20 spacji). Po nazwie postaci następuje jej kwestia dialogowa.
- Po nazwie postaci mogą występować didaskalia w nawiasach, np. `(V.O.)`, `(O.S.)`, `(CONT'D)` — te fragmenty trzeba usunąć, aby uzyskać czyste imię.

Algorytm czyta tekst linijka po linijce: gdy napotka nagłówek sceny, otwiera nową scenę; gdy napotka nazwę postaci, dodaje ją do bieżącej sceny. Wynikiem jest lista scen, z których każda zawiera zbiór unikalnych postaci.

In [ ]:
"""
PROMPT DLA AI:
Napisz skrypt, który przetwarza tekst scenariusza filmowego (zapisany w zmiennej
tekstowej) i wyodrębnia z niego strukturę scen oraz postaci.

Logika działania:
1. Podziel tekst na pojedyncze linie.
2. Rozpoznaj nagłówek nowej sceny po tym, że linia (po usunięciu wiodących
   spacji) zaczyna się od 'INT.' lub 'EXT.'.
3. Rozpoznaj nazwę postaci po tym, że:
   a) linia jest mocno wcięta od lewej (co najmniej 20 spacji na początku),
   b) tekst po usunięciu wcięcia jest zapisany WIELKIMI LITERAMI,
   c) tekst nie jest pusty i nie jest typową didaskaliją (nie zaczyna się
      od nawiasu otwierającego).
4. Z rozpoznanej nazwy postaci usuń didaskalia w nawiasach, np. '(V.O.)',
   '(O.S.)', '(CONT\'D)' itp.
5. Odfiltruj artefakty: nazwy krótsze niż 2 znaki oraz typowe słowa
   kluczowe scenariusza, które nie są postaciami (np. 'FADE IN',
   'FADE OUT', 'CUT TO', 'CONTINUED', 'THE END', 'DISSOLVE TO',
   'ANGLE ON', 'CLOSE ON', 'BACK TO', 'INTERCUT', 'FLASHBACK',
   'LATER', 'TITLE CARD', 'SUPER', 'MONTAGE').
6. Wynikiem ma być lista, gdzie każdy element odpowiada jednej scenie
   i zawiera: numer sceny, nagłówek sceny oraz zbiór unikalnych postaci
   w tej scenie.
7. Wyświetl liczbę znalezionych scen oraz podgląd pierwszych 10 scen
   z przypisanymi postaciami.
"""

import re

# Słowa kluczowe scenariusza, które NIE są postaciami
WYKLUCZENIA = {
    "FADE IN", "FADE OUT", "FADE TO BLACK", "CUT TO", "CUT BACK TO",
    "DISSOLVE TO", "CONTINUED", "CONTINUOUS", "THE END", "MORE",
    "ANGLE ON", "CLOSE ON", "BACK TO", "INTERCUT", "FLASHBACK",
    "LATER", "TITLE CARD", "SUPER", "MONTAGE", "SMASH CUT TO",
    "TIME CUT", "MATCH CUT TO", "CREDITS", "END CREDITS",
    "OPENING CREDITS", "RESUME"
}

def wyodrebnij_sceny(tekst):
    """Parsuje tekst scenariusza i zwraca listę scen z postaciami."""
    linie = tekst.split("\n")
    sceny = []
    biezaca_scena = None

    for linia in linie:
        linia_czysta = linia.strip()

        # --- Rozpoznanie nagłówka sceny ---
        if linia_czysta.startswith("INT.") or linia_czysta.startswith("EXT.") or \
           linia_czysta.startswith("INT/EXT.") or linia_czysta.startswith("EXT/INT.") or \
           linia_czysta.startswith("INT ") or linia_czysta.startswith("EXT "):
            biezaca_scena = {
                "naglowek": linia_czysta,
                "postacie": set()
            }
            sceny.append(biezaca_scena)
            continue

        # --- Rozpoznanie nazwy postaci ---
        if biezaca_scena is None:
            continue

        # Sprawdź wcięcie (co najmniej 20 spacji)
        wcięcie = len(linia) - len(linia.lstrip(" "))
        if wcięcie < 20:
            continue

        # Tekst po usunięciu wcięcia
        kandydat = linia_czysta

        # Pomiń puste linie i didaskalia (zaczynające się od nawiasu)
        if not kandydat or kandydat.startswith("("):
            continue

        # Usuń didaskalia w nawiasach z nazwy postaci
        nazwa = re.sub(r"\(.*?\)", "", kandydat).strip()

        # Sprawdź, czy tekst jest zapisany wielkimi literami
        if not nazwa or nazwa != nazwa.upper():
            continue

        # Sprawdź, czy to nie jest słowo kluczowe scenariusza
        if nazwa in WYKLUCZENIA:
            continue

        # Sprawdź minimalną długość i czy zawiera litery
        if len(nazwa) < 2 or not any(c.isalpha() for c in nazwa):
            continue

        biezaca_scena["postacie"].add(nazwa)

    return sceny


# Uruchomienie parsowania
sceny = wyodrebnij_sceny(tekst_scenariusza)

print(f"Znaleziono {len(sceny)} scen.")
print("=" * 60)

# Podgląd pierwszych 10 scen
for i, scena in enumerate(sceny[:10], start=1):
    postacie_str = ", ".join(sorted(scena["postacie"])) if scena["postacie"] else "(brak postaci)"
    print(f"\nScena {i}: {scena['naglowek'][:70]}")
    print(f"  Postacie: {postacie_str}")

---
## Etap 3: Częstotliwość występowania postaci

Zanim przejdziemy do analizy relacji, warto sprawdzić, **kto jest najważniejszy** w scenariuszu — czyli w ilu scenach pojawia się każda z postaci. To podstawowa miara znaczenia postaci w fabule.

Wynik przedstawimy jako wykres słupkowy 10 najczęściej występujących postaci.

In [ ]:
"""
PROMPT DLA AI:
Na podstawie listy scen (gdzie każda scena zawiera zbiór postaci) oblicz,
w ilu scenach łącznie pojawia się każda postać. Następnie:
1. Posortuj postacie od najczęściej do najrzadziej występującej.
2. Wyświetl tabelę z nazwą postaci i liczbą scen.
3. Wygeneruj czytelny, poziomy wykres słupkowy dla 10 postaci
   o największej liczbie wystąpień. Wykres ma mieć tytuł i opisy osi.
"""

from collections import Counter
import matplotlib.pyplot as plt

# Zliczenie wystąpień każdej postaci we wszystkich scenach
licznik_postaci = Counter()
for scena in sceny:
    for postac in scena["postacie"]:
        licznik_postaci[postac] += 1

# Posortowana lista
ranking = licznik_postaci.most_common()

print(f"Łącznie znaleziono {len(ranking)} unikalnych postaci.\n")
print(f"{'Postać':<30} {'Liczba scen':>12}")
print("-" * 43)
for postac, liczba in ranking[:20]:
    print(f"{postac:<30} {liczba:>12}")

# Wykres słupkowy — 10 najczęstszych postaci
top10 = ranking[:10]
nazwy = [p for p, _ in top10][::-1]  # odwrócenie dla czytelności
liczby = [n for _, n in top10][::-1]

plt.figure(figsize=(10, 6))
plt.barh(nazwy, liczby, color="steelblue")
plt.xlabel("Liczba scen")
plt.ylabel("Postać")
plt.title("10 najczęściej występujących postaci w scenariuszu")
plt.tight_layout()
plt.show()

---
## Etap 4: Budowa relacji — współwystąpienia w scenach

Teraz budujemy właściwą **sieć powiązań**. Logika jest prosta:

- Jeśli w jednej scenie występują postacie A i B, tworzymy między nimi **krawędź** (połączenie).
- Jeśli A i B pojawiają się razem w wielu scenach, krawędź ma wyższą **wagę** — relacja jest silniejsza.

Dla każdej sceny generujemy wszystkie możliwe pary postaci, a następnie zliczamy, ile razy każda para wystąpiła w całym scenariuszu. Wynik zapisujemy w formie tabeli: *postać A → postać B → waga*.

In [ ]:
"""
PROMPT DLA AI:
Na podstawie listy scen (gdzie każda scena zawiera zbiór postaci) zbuduj
tabelę współwystąpień:
1. Dla każdej sceny wygeneruj wszystkie możliwe pary postaci (bez powtórzeń,
   kolejność w parze alfabetyczna).
2. Zlicz, ile razy każda para występuje w całym scenariuszu — to będzie
   waga krawędzi.
3. Zapisz wynik jako ustrukturyzowaną tabelę z trzema kolumnami:
   'Zrodlo' (postać A), 'Cel' (postać B), 'Waga'.
4. Posortuj tabelę od najsilniejszych do najsłabszych relacji.
5. Wyświetl 15 najsilniejszych relacji.
"""

from itertools import combinations
import pandas as pd

# Generowanie par współwystąpień
licznik_par = Counter()

for scena in sceny:
    postacie_lista = sorted(scena["postacie"])
    if len(postacie_lista) < 2:
        continue
    for para in combinations(postacie_lista, 2):
        licznik_par[para] += 1

# Budowa tabeli
wiersze = []
for (postac_a, postac_b), waga in licznik_par.most_common():
    wiersze.append({"Zrodlo": postac_a, "Cel": postac_b, "Waga": waga})

df_krawedzie = pd.DataFrame(wiersze)

print(f"Łącznie {len(df_krawedzie)} unikalnych par (krawędzi).\n")
print("15 najsilniejszych relacji:")
print("=" * 50)
print(df_krawedzie.head(15).to_string(index=False))

---
## Etap 5: Wizualizacja sieci i eksport danych

Na koniec wykonujemy dwie rzeczy:

1. **Podgląd grafu** — prosty rysunek sieci powiązań, który pozwala rzucić okiem na strukturę relacji bezpośrednio w notatniku.
2. **Eksport do plików** — to najważniejszy krok z perspektywy dalszej analizy. Zapisujemy dane w dwóch formatach:
   - **CSV** — uniwersalny plik arkusza kalkulacyjnego, który można otworzyć w Excelu, Google Sheets lub zaimportować do dowolnego narzędzia.
   - **GraphML** — specjalistyczny format XML do opisu grafów, natywnie obsługiwany przez programy do analizy sieci, takie jak Gephi.

In [ ]:
"""
PROMPT DLA AI:
Na podstawie tabeli relacji (z kolumnami: Zrodlo, Cel, Waga) wykonaj
następujące kroki:

1. Zbuduj obiekt grafu (sieć) z tych danych — węzłami są postacie,
   krawędziami są ich relacje, a każda krawędź ma atrybut wagi.
2. Narysuj podstawowy wizualny podgląd tego grafu. Rozmiar węzłów
   powinien zależeć od liczby połączeń danej postaci (im więcej,
   tym większy węzeł). Grubość krawędzi powinna odpowiadać wadze.
   Dodaj etykiety z nazwami postaci. Użyj algorytmu rozmieszczającego
   węzły tak, aby powiązane postacie były bliżej siebie.
3. Zapisz tabelę krawędzi do pliku arkusza kalkulacyjnego (CSV)
   o nazwie 'krawedzie.csv'.
4. Zapisz cały graf (z wagami krawędzi) do pliku w formacie sieciowym
   GraphML o nazwie 'graf_postaci.graphml'.
5. Wyświetl potwierdzenie zapisania plików.
"""

import networkx as nx
import matplotlib.pyplot as plt

# --- Budowa grafu ---
G = nx.Graph()

for _, wiersz in df_krawedzie.iterrows():
    G.add_edge(wiersz["Zrodlo"], wiersz["Cel"], weight=wiersz["Waga"])

print(f"Graf: {G.number_of_nodes()} węzłów, {G.number_of_edges()} krawędzi.\n")

# --- Wizualizacja ---
plt.figure(figsize=(14, 10))

# Rozmieszczenie węzłów algorytmem sprężynowym
pozycje = nx.spring_layout(G, k=0.8, iterations=50, seed=42)

# Rozmiar węzłów proporcjonalny do stopnia (liczby połączeń)
stopnie = dict(G.degree())
rozmiary = [stopnie[wezel] * 40 + 50 for wezel in G.nodes()]

# Grubość krawędzi proporcjonalna do wagi
wagi = [G[u][v]["weight"] for u, v in G.edges()]
max_waga = max(wagi) if wagi else 1
grubosci = [0.3 + (w / max_waga) * 3 for w in wagi]

# Rysowanie
nx.draw_networkx_edges(G, pozycje, width=grubosci, alpha=0.3, edge_color="gray")
nx.draw_networkx_nodes(G, pozycje, node_size=rozmiary, node_color="steelblue", alpha=0.7)
nx.draw_networkx_labels(G, pozycje, font_size=7, font_weight="bold")

plt.title("Sieć współwystąpień postaci w scenariuszu", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.show()

# --- Eksport do CSV ---
sciezka_csv = "krawedzie.csv"
df_krawedzie.to_csv(sciezka_csv, index=False, encoding="utf-8-sig")
print(f"Zapisano tabelę krawędzi: {sciezka_csv}")

# --- Eksport do GraphML ---
sciezka_graphml = "graf_postaci.graphml"
nx.write_graphml(G, sciezka_graphml)
print(f"Zapisano graf w formacie GraphML: {sciezka_graphml}")

print("\nPliki gotowe do pobrania (użyj panelu plików po lewej stronie w Colab).")

---
## Co dalej?

- **Otwórz `graf_postaci.graphml` w Gephi** — program pozwala interaktywnie eksplorować sieć, obliczać miary centralności (np. pośrednictwo, bliskość), wykrywać społeczności i tworzyć publikowalne wizualizacje.
- **Otwórz `krawedzie.csv` w Excelu lub Google Sheets** — możesz filtrować, sortować i dalej analizować relacje między postaciami.
- **Podmień URL** w Etapie 1 na inny scenariusz z IMSDb i uruchom cały notatnik ponownie, żeby porównać struktury narracyjne różnych filmów.